In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
customers = spark.table(
    "ecommerce.silver.customers"
)

products = spark.table(
    "ecommerce.silver.products"
)

orders = spark.table(
    "ecommerce.silver.orders"
)

In [0]:
sales = (
    orders
    .join(
        customers,
        orders.customer_id == customers.customer_id,
        "inner")
    .join(
        products,
        orders.product_id == products.product_id,
        "inner")
    .withColumn(
        "sales_amount",
        col("quantity")*col("price"))
    
    .select(
    col('orders.order_id'),
    orders.order_date,
    col('orders.customer_id'),
    customers.state,
    customers.customer_name,
    products.product_name,
    products.category,
    orders.quantity.cast("double"),
    products.price.cast("double"),
    col("sales_amount").cast("double")
)

    )

sales.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce.gold.sales")



In [0]:
sales.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save( "abfss://inputdata@storagelake9720.dfs.core.windows.net/gold/sales")


In [0]:

daily_sales = (
    sales
    .groupBy("order_date")
    .agg(
        sum("sales_amount").alias("total_sales")
    )
    .orderBy("order_date")
)

daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce.gold.daily_sales_summary")


In [0]:
state_sales = (
    sales
    .join(
        customers,
        sales.customer_id == customers.customer_id,
        "inner"
    )
    .groupBy("category")
    .agg(
        sum("sales_amount").alias("total_sales")
    )
)


state_sales.write.mode("overwrite").saveAsTable("ecommerce.gold.state_sales")